# Plotly in dash_server

This notebook is an example how to run a dashboard server with plotly

In [ ]:
import os
import numpy as np
import pandas as pd

import dash
from dash import Dash, dcc, html, Input, Output, callback
#import dash_bootstrap_components as dbc
import plotly.graph_objects as go
import plotly.express as px

from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Necessary variables
REPORT_URL = os.getenv("REPORT_URL")
REPORT_PORT = os.getenv("REPORT_PORT")
HOSTNAME = os.getenv("HOSTNAME")
SERVERNAME = os.getenv("SERVERNAME")
server_url=f"https://{SERVERNAME}/{REPORT_URL}"

print(f'You will be able to access your report at {server_url}')

# Create a Dashboard object
#app = dash.Dash(__name__, external_stylesheets=[dbc.themes.SANDSTONE], url_base_pathname="/"+REPORT_URL)
app = dash.Dash(__name__, url_base_pathname="/"+REPORT_URL)


In [ ]:
# Data
DATADIR = os.getenv("DATADIR")

df = pd.read_csv('https://raw.githubusercontent.com/plotly/datasets/master/gapminderDataFiveYear.csv')
df.head()

In [ ]:
# Define the dashboard layout
app.layout = html.Div([
    dcc.Graph(id='graph-with-slider'),
    dcc.Slider(
        df['year'].min(),
        df['year'].max(),
        step=None,
        value=df['year'].min(),
        marks={str(year): str(year) for year in df['year'].unique()},
        id='year-slider'
    )
])

# When we change anything on the slider it will call this function to update the plot
@callback(
    Output('graph-with-slider', 'figure'),
    Input('year-slider', 'value'))
def update_figure(selected_year):
    filtered_df = df[df.year == selected_year]

    fig = px.scatter(filtered_df, x="gdpPercap", y="lifeExp",
                     size="pop", color="continent", hover_name="country",
                     log_x=True, size_max=55)

    fig.update_layout(transition_duration=500)

    return fig

In [ ]:
# Starts a server
print(f"Visit {server_url}")
app.run(debug=False, port=REPORT_PORT, host=HOSTNAME)